# Probabilistic programming languages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/languages.ipynb)

`symprog` aims to make it simple to implement mini-probabilistic programming languages, and comes with two example languages that are written *inline* in Python, as the body of a function decorated with `@Parser`. The function body is never run as ordinary Python. Instead, it is parsed and compiled in two ways:

- into a **solver** (see `solver_tutorial.ipynb`), for exact inference;
- into a plain **Python sampling program**, which spells out what the program *means* by simulating it.

This notebook covers:

1. The **constraint language**: weighted variables and Boolean constraints.
2. **Probabilistic logic programs** (PLP): probabilistic facts and logical rules, in the style of [ProbLog](https://dtai.cs.kuleuven.be/problog/).

## Setup

If `symprog` is not installed, the notebook uses the version in this repository.

In [1]:
# %pip install -q "symprog[notebooks]"

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

from IPython.display import Code

from symprog import Parser, Solver

# The constraint language

A constraint program declares **weighted variables** (`a: 0.9` means $a$ is true with probability $0.9$) followed by **constraints**, one per line, written with Python's `and`, `or` and `not` or with `And`, `Or`, `Not`, `Implies`, `Biconditional` and `Xor`. Inference computes the probability that all constraints hold:

In [2]:
@Parser(language="constraint")
def prog():
    ## variable weights
    a: 0.9; b: 0.7; c: 0.65

    ## Boolean constraints
    (not a) or b
    c or b
    (not b) or c

prog.inference().inference

tensor(0.4745)

#### What does a program mean?

The meaning of a program is given by the Python program it compiles to, `prog.prog_str`: sample each variable according to its weight, then check whether all constraints hold:

In [3]:
Code(prog.prog_str, language="python")

def prog():
    a = np.random.random() < 0.9
    b = np.random.random() < 0.7
    c = np.random.random() < 0.65
    r1 = not a or b
    r2 = c or b
    r3 = not b or c
    if r1 and r2 and r3:
        return 'True'
    else:
        return 'False'

Calling the program runs this simulation once, and `sample` runs it many times. The fraction of runs in which the constraints hold estimates the exact probability above:

In [4]:
print("one run:", prog())

samples = prog.sample(k=10000)
estimate = samples["return_values"]["True"] / samples["executed"]
print(f"sampling estimate: {estimate:.3f}   exact: {prog.inference().inference.item():.3f}")

one run: False
sampling estimate: 0.472   exact: 0.475


#### Under the hood

Exact inference is done by an ordinary `Solver`, available as `prog.solver`. It is equivalent to adding the constraints and weights to a solver by hand:

In [5]:
print(prog.solver)

solver = Solver()
solver.add(["Or(Not(a), b)", "Or(c, b)", "Or(Not(b), c)"])
solver.add_weights({"a": 0.9, "b": 0.7, "c": 0.65})
assert abs(solver.query().inference.item() - prog.inference().inference.item()) < 1e-6

Solver(
  num_formulas=3,
  weight_backend=PytorchWeights,
  solver_backend=SddMapper,
  conditional=False,
)


#### Conditioning

A `condition:` statement conditions on a formula, computing $Pr(\text{constraints} \mid \text{condition})$. In the sampling program it becomes an `assert`, i.e., [rejection sampling](https://en.wikipedia.org/wiki/Rejection_sampling): runs that violate the condition are discarded.

In [6]:
@Parser(language="constraint")
def conditioned():
    a: 0.9; b: 0.7; c: 0.65

    (not a) or b
    c or b
    (not b) or c

    condition: a

## the same computation with a solver: P(F | a)
solver.add_conditional("a")
assert abs(solver.query().inference.item() - conditioned.inference().inference.item()) < 1e-6
solver.remove_conditional()

Code(conditioned.prog_str, language="python")

def conditioned():
    a = np.random.random() < 0.9
    b = np.random.random() < 0.7
    c = np.random.random() < 0.65
    r1 = not a or b
    r2 = c or b
    r3 = not b or c
    assert a
    if r1 and r2 and r3:
        return 'True'
    else:
        return 'False'

#### Operators

`And` and `Or` take any number of arguments, and `Xor` means *exactly one* argument is true:

In [7]:
@Parser(language="constraint")
def exactly_one():
    a: 0.9; b: 0.7; c: 0.5
    Xor(a, b, c)

expected = 0.9 * 0.3 * 0.5 + 0.1 * 0.7 * 0.5 + 0.1 * 0.3 * 0.5
assert abs(exactly_one.inference().inference.item() - expected) < 1e-6
exactly_one.inference().inference

tensor(0.1850)

#### First-order constraints

Constraints can also be stated over a *domain* of constants, using *predicates*:

- `a: X` declares a constant `a` of type `X`;
- `s: {a: 0.9, b: 0.7}` gives the weights of the predicate `s` for each constant (predicates without weights default to $0.5$);
- a constraint like `Implies(s(X), f(X))` is *grounded*, i.e., stated once for each constant of type `X`;
- an aggregate like `Or(s(X))` combines the grounded atoms into a single constraint ($s(a) \lor s(b) \lor s(c)$).

Below, every `s` must also be an `f`, and at least one constant must be an `s`:

In [8]:
@Parser(language="constraint")
def first_order():
    a: X; b: X; c: X
    s: {a: 0.9, b: 0.7, c: 0.2}
    f: {a: 0.5, b: 0.5, c: 0.5}

    Implies(s(X), f(X))
    Or(s(X))

print(first_order.inference().inference)
Code(first_order.prog_str, language="python")

tensor(0.2977)


def first_order():
    s__a = np.random.random() < 0.9
    s__b = np.random.random() < 0.7
    s__c = np.random.random() < 0.2
    f__a = np.random.random() < 0.5
    f__b = np.random.random() < 0.5
    f__c = np.random.random() < 0.5
    r1 = not s__a or f__a
    r2 = not s__b or f__b
    r3 = not s__c or f__c
    r4 = s__a or s__b or s__c
    if r1 and r2 and r3 and r4:
        return 'True'
    else:
        return 'False'

#### Other inferences

Programs support all of the solver's inference modes, symbolic computation and gradients (see `solver_tutorial.ipynb` and `symbolic_and_gradients.ipynb`):

In [9]:
print("Viterbi: ", prog.inference(mode="Viterbi").inference)
print("Symbolic:", prog.inference(mode="Prob", symbolic=True).inference)
print("Gradient:", dict(zip(prog.solver.weight_names, prog.grad(key="weights").tolist())))

Viterbi:  tensor(0.4095)
Symbolic: a*b*c + c*(1 - a)
Gradient: {'a': -0.19499999284744263, 'b': 0.5849999785423279, 'c': 0.7300000190734863}


# Probabilistic logic programs

The `plp` language implements probabilistic logic programming inspired by [ProbLog](https://dtai.cs.kuleuven.be/problog/). A program consists of:

- **probabilistic facts**, e.g., `heads1: 0.5`;
- **rules** `head :- body`, where the body combines atoms with `&` (and), `|` (or) and `~` (not);
- **queries** `query(atom)` and **evidence** `evidence(atom, True)`.

Its meaning: sample the facts, then derive everything that follows from the rules. The probability of a query is the probability that it is derived. For example, with two coins:

In [10]:
@Parser(language="plp")
def coins():
    heads1: 0.5
    heads2: 0.6

    twoHeads :- heads1 & heads2
    someHeads :- heads1
    someHeads :- heads2

    query(twoHeads)
    query(someHeads)

for result in coins.inference():
    print(result.details["marginal"], result.inference.item())

someHeads 0.800000011920929
twoHeads 0.30000001192092896


With several queries, `inference` returns one result per query (sorted by name). Note how the two rules for `someHeads` act as a disjunction: $Pr(\text{someHeads}) = 1 - 0.5 \times 0.4 = 0.8$. The sampling program makes this explicit:

In [11]:
Code(coins.python_prog, language="python")

def coins():
    heads1 = np.random.random() < 0.5
    heads2 = np.random.random() < 0.6
    twoHeads = heads1 and heads2
    someHeads = heads1 or heads2
    return {'twoHeads': twoHeads, 'someHeads': someHeads}

#### Probabilistic rules and evidence

A rule with a weight, `e[0.3] :- c1`, fires with probability $0.3$ when its body holds. Several such rules for the same head give a *noisy-or*. Evidence is given with `evidence(atom, True/False)`:

In [12]:
@Parser(language="plp")
def noisy_or():
    c1: 0.5; c2: 0.5

    e[0.3] :- c1
    e[0.4] :- c2

    evidence(c1, True)
    evidence(c2, True)
    query(e)

## given both causes, e fails only if neither rule fires
assert abs(noisy_or.inference().inference.item() - (1 - 0.7 * 0.6)) < 1e-6
noisy_or.inference().inference

tensor(0.5800)

Evidence can also be passed to `inference`, e.g., `noisy_or.inference(evidence="~c2")`.

#### Negation

`~a` in a rule body means that `a` is *not* derived (negation as failure). A rule with a negative head, `b[False] :- c`, is *inhibitory*: it prevents `b` from being derived when `c` holds:

In [13]:
@Parser(language="plp")
def inhibition():
    a: 0.9; c: 0.5

    b :- a
    b[False] :- c    ## c inhibits b

    query(b)

assert abs(inhibition.inference().inference.item() - 0.9 * 0.5) < 1e-6
Code(inhibition.python_prog, language="python")

def inhibition():
    a = np.random.random() < 0.9
    c = np.random.random() < 0.5
    b = a and (not c)
    return {'b': b}

#### Annotated disjunctions

An *annotated disjunction* `one[1/3] | two[1/3] | three[1/3]` chooses at most one of its atoms, with the given probabilities (if they sum to less than one, possibly none of them). For example, a three-sided die:

In [14]:
@Parser(language="plp")
def die():
    one[1/3] | two[1/3] | three[1/3]
    low :- one | two
    query(low)

assert abs(die.inference().inference.item() - 2 / 3) < 1e-6
die.inference().inference

tensor(0.6667)

#### First-order programs

Atoms can take arguments, written in square brackets: lowercase names are constants (`person[john]`) and uppercase names are variables (`X`). Rules with variables are grounded for every constant. For example, the [alarm network](https://dtai.cs.kuleuven.be/problog/tutorial/basic/02_bayes.html) from the ProbLog tutorial, where `calls[X, 0.8] :- ...` is a probabilistic rule for each person `X`:

In [15]:
@Parser(language="plp")
def alarm():
    person[john]: 1.0; person[mary]: 1.0
    burglary: 0.7; earthquake: 0.2

    alarm[0.9] :- burglary & earthquake
    alarm[0.8] :- burglary & ~earthquake
    alarm[0.1] :- ~burglary & earthquake

    calls[X, 0.8] :- alarm & person[X]
    calls[X, 0.1] :- ~alarm & person[X]

    evidence(calls[john], True)
    evidence(calls[mary], True)
    query(burglary)

## ProbLog gives 0.9819 for this program
assert abs(alarm.inference().inference.item() - 0.98194) < 1e-4
alarm.inference().inference

tensor(0.9819)

Grounded atoms are named with double underscores, e.g., `calls[john]` becomes the variable `calls__john`, which can be used in evidence and queries passed to `inference`:

In [16]:
alarm.inference(evidence="~calls__mary").inference

tensor(0.9324)

#### Recursion

Rules can be recursive. Below is [probabilistic graph reachability](https://dtai.cs.kuleuven.be/problog/tutorial/basic/04_pgraph.html): each edge is present with some probability, and we ask for the probability that there is a path from `n1` to `n5`:

In [17]:
@Parser(language="plp")
def graph():
    edge[n1, n2]: 0.6
    edge[n1, n3]: 0.1
    edge[n2, n5]: 0.4
    edge[n2, n6]: 0.3
    edge[n3, n4]: 0.3
    edge[n4, n5]: 0.8
    edge[n5, n6]: 0.2

    path[X, Y] :- edge[X, Y]
    path[X, Y] :- edge[X, Z] & path[Z, Y]

    query(path[n1, n5])

## ProbLog gives 0.25824 for this program
assert abs(graph.inference().inference.item() - 0.25824) < 1e-5
graph.inference().inference

tensor(0.2582)

The sampling program shows how the recursive rules are grounded into derivations of each path:

In [18]:
Code(graph.python_prog, language="python")

def graph():
    edge__n1__n2 = np.random.random() < 0.6
    edge__n1__n3 = np.random.random() < 0.1
    edge__n2__n5 = np.random.random() < 0.4
    edge__n2__n6 = np.random.random() < 0.3
    edge__n3__n4 = np.random.random() < 0.3
    edge__n4__n5 = np.random.random() < 0.8
    edge__n5__n6 = np.random.random() < 0.2
    path__n1__n2 = edge__n1__n2
    path__n1__n3 = edge__n1__n3
    path__n3__n4 = edge__n3__n4
    path__n1__n4 = edge__n1__n3 and path__n3__n4
    path__n4__n5 = edge__n4__n5
    path__n3__n5 = edge__n3__n4 and path__n4__n5
    path__n2__n5 = edge__n2__n5
    path__n1__n5 = edge__n1__n2 and path__n2__n5 or (edge__n1__n3 and path__n3__n5)
    path__n5__n6 = edge__n5__n6
    path__n2__n6 = edge__n2__n6 or (edge__n2__n5 and path__n5__n6)
    path__n4__n6 = edge__n4__n5 and path__n5__n6
    path__n3__n6 = edge__n3__n4 and path__n4__n6
    path__n1__n6 = edge__n1__n2 and path__n2__n6 or (edge__n1__n3 and path__n3__n6)
    return {'path__n1__n5': path__n1__n5}

Gradients tell us which edges matter for the query, and by how much. The edges into `n6` cannot be part of a path to `n5`, so their gradients are zero:

In [19]:
{name: round(g, 4) for name, g in zip(graph.solver.weight_names, graph.grad(key="weights").tolist())}

{'edge__n1__n2': 0.3904,
 'edge__n1__n3': 0.1824,
 'edge__n2__n5': 0.5856,
 'edge__n2__n6': 0.0,
 'edge__n3__n4': 0.0608,
 'edge__n4__n5': 0.0228,
 'edge__n5__n6': 0.0}

#### Limitations

The language covers a core of ProbLog (facts, rules, negation, evidence, annotated disjunctions, first-order rules and recursion), but not, e.g., arithmetic or built-in predicates in rule bodies, lists, or probabilities computed at runtime. The ProbLog examples that are and are not supported are listed in [`tests/test_plp_problog.py`](https://github.com/TuLLI-AI/symprog/blob/main/tests/test_plp_problog.py).

## Next steps

- **Solver tutorial** (`solver_tutorial.ipynb`): the solvers behind these programs.
- **Symbolic computation and gradients** (`symbolic_and_gradients.ipynb`) and **tensorization** (`tensorization_and_jax.ipynb`), which apply to programs via `prog.solver`.